In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab data access)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Disaster Triage: 3-Class RED / YELLOW / GREEN with 42 Curated Features (`models/train_disaster_triage_3class_exp.ipynb`)

Trains the Disaster Triage model using the **42 curated features from `train_oof_logistic_regression_stacking_exp.ipynb`** (vitals, ranges, ratios, threshold derangements, and chief complaint interactions) loaded directly from **`datasets/5v_cleandf.RData`**.

### 🏥 3-Class Disaster Triage Mapping
- **`RED` (Class 0)**: Immediate / Resuscitation (`ESI 1-2`)
- **`YELLOW` (Class 1)**: Delayed / Urgent (`ESI 3`)
- **`GREEN` (Class 2)**: Minimal / Minor (`ESI 4-5`)

### 🔬 Features & Clinical Safety Objectives
- **42 Features**: 12 base raw measurements (including blood pressure `sbp`/`dbp`) + 30 clinically derived engineered features (ranges, shock index, NEWS-like composites, CC interactions, squared deviations).
- **Safety Thresholding**: Tuned for **RED Sensitivity $\ge 90\%$** on the validation set, ensuring **undertriage < 5%** according to START/SALT doctrine.
- **Primary Metric**: Evaluates on **Balanced Accuracy** and per-class **Sensitivity / Recall**.

In [ ]:
%%R
# ---------------------------------------------------------------------------
# Step 1: Load 5v_cleandf.RData and Extract 16 Base Features for 42-Feature Pipeline
# ---------------------------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
})

data_file <- "../datasets/5v_cleandf.RData"
if (!file.exists(data_file)) data_file <- "datasets/5v_cleandf.RData"

data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)

cat(sprintf("Loaded %s: %d rows, %d columns\n", data_file, nrow(raw_df), ncol(raw_df)))

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(is.na(raw_df$gender), NA, ifelse(as.character(raw_df$gender) == "Male", 1, 0)) else rep(NA, nrow(raw_df))
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) raw_df$cc_breathingdifficulty else rep(NA, nrow(raw_df))

get_vec <- function(col_name) {
  if (col_name %in% names(raw_df)) {
    return(raw_df[[col_name]])
  } else {
    return(rep(NA, nrow(raw_df)))
  }
}

raw_esi_char <- as.character(raw_df$esi)

df_master <- data.frame(
  age                     = raw_df$age,
  cc_breathingdifficulty  = cc_bd_vec,
  gender                  = gender_vec,
  triage_vital_hr         = get_vec("triage_vital_hr"),
  triage_vital_sbp        = get_vec("triage_vital_sbp"),
  triage_vital_dbp        = get_vec("triage_vital_dbp"),
  triage_vital_rr         = get_vec("triage_vital_rr"),
  triage_vital_o2         = get_vec("triage_vital_o2"),
  pulse_min               = get_vec("pulse_min"),
  resp_min                = get_vec("resp_min"),
  spo2_min                = get_vec("spo2_min"),
  sbp_min                 = get_vec("sbp_min"),
  pulse_max               = get_vec("pulse_max"),
  resp_max                = get_vec("resp_max"),
  spo2_max                = get_vec("spo2_max"),
  sbp_max                 = get_vec("sbp_max"),
  esi                     = raw_esi_char
)

# Strictly drop rows with >= 1 NA across the 16 base columns
df_master <- na.omit(df_master)
raw_mat_export <- as.matrix(df_master[, 1:16])
esi_export     <- as.numeric(as.character(df_master$esi))

cat(sprintf("Cleaned Matrix Exported: %d Complete Cases, 16 Base Columns\n", nrow(raw_mat_export)))

In [ ]:
# ---------------------------------------------------------------------------
# Step 2: Build 42 Curated Features & Partition Train/Val/Test
# ---------------------------------------------------------------------------
import json, os, pickle
from rpy2.robjects import r
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, cohen_kappa_score)

ROOT = '..' if os.path.basename(os.getcwd()) == 'models' else '.'

raw_mat_in = np.array(r('raw_mat_export'), dtype=np.float64)
esi        = np.array(r('esi_export'), dtype=np.int32)

def build_42_feature_matrix(raw_mat):
    N = len(raw_mat)
    X = np.zeros((N, 42), dtype=np.float32)
    
    # Extract base columns
    age       = raw_mat[:, 0]
    cc_bd     = raw_mat[:, 1]
    gender    = raw_mat[:, 2]
    t_hr      = raw_mat[:, 3]
    t_sbp     = raw_mat[:, 4]
    t_dbp     = raw_mat[:, 5]
    t_rr      = raw_mat[:, 6]
    t_o2      = raw_mat[:, 7]
    pulse_min = raw_mat[:, 8]
    resp_min  = raw_mat[:, 9]
    spo2_min  = raw_mat[:, 10]
    sbp_min   = raw_mat[:, 11]
    pulse_max = raw_mat[:, 12]
    resp_max  = raw_mat[:, 13]
    spo2_max  = raw_mat[:, 14]
    sbp_max   = raw_mat[:, 15]
    
    hr_rng   = pulse_max - pulse_min
    rr_rng   = resp_max - resp_min
    spo2_rng = spo2_max - spo2_min
    sbp_rng  = sbp_max - sbp_min
    
    # 1..12: Specified Raw Features
    X[:, 0]  = age
    X[:, 1]  = cc_bd
    X[:, 2]  = gender
    X[:, 3]  = t_hr
    X[:, 4]  = t_sbp
    X[:, 5]  = t_dbp
    X[:, 6]  = t_rr
    X[:, 7]  = pulse_min
    X[:, 8]  = resp_min
    X[:, 9]  = spo2_min
    X[:, 10] = pulse_max
    X[:, 11] = spo2_max
    
    # 13..22: Baseline Threshold Flags
    is_dyspnea_tot  = (t_o2 < 90).astype(float)
    is_dyspnea_mod  = ((t_o2 >= 90) & (t_o2 < 94)).astype(float)
    is_brady_pnea   = (t_rr < 10).astype(float)
    is_tachy_pnea   = (t_rr > 30).astype(float)
    is_hypo_tension = (t_sbp <= 90).astype(float)
    is_hyper_tension= (t_sbp > 220).astype(float)
    is_brady_tot    = (t_hr < 40).astype(float)
    is_brady_mod    = ((t_hr >= 40) & (t_hr < 60)).astype(float)
    is_tachy_tot    = (t_hr > 150).astype(float)
    is_tachy_mod    = ((t_hr >= 100) & (t_hr <= 150)).astype(float)
    
    X[:, 12] = is_dyspnea_tot
    X[:, 13] = is_dyspnea_mod
    X[:, 14] = is_brady_pnea
    X[:, 15] = is_tachy_pnea
    X[:, 16] = is_hypo_tension
    X[:, 17] = is_hyper_tension
    X[:, 18] = is_brady_tot
    X[:, 19] = is_brady_mod
    X[:, 20] = is_tachy_tot
    X[:, 21] = is_tachy_mod
    
    # 23..35: Ranges, Mid-to-Triage, Ratios
    X[:, 22] = hr_rng
    X[:, 23] = rr_rng
    X[:, 24] = spo2_rng
    X[:, 25] = sbp_rng
    shock_idx = t_hr / np.where(t_sbp == 0, 1.0, t_sbp)
    X[:, 26] = shock_idx
    X[:, 27] = t_hr - hr_rng
    X[:, 28] = t_sbp - sbp_rng
    X[:, 29] = t_rr - rr_rng
    X[:, 30] = t_o2 - spo2_rng
    X[:, 31] = t_o2 / np.where(t_rr == 0, 1.0, t_rr) # rox_index
    X[:, 32] = spo2_rng / np.where(spo2_max == 0, 1.0, spo2_max) # spo2_drop_ratio
    X[:, 33] = hr_rng / (t_hr + 1.0) # hr_instability_ratio
    X[:, 34] = (t_rr / np.where(t_o2 == 0, 1.0, t_o2)) * 100.0 # bif
    
    # 36..42: Curated Advanced Features
    X[:, 35] = (is_dyspnea_tot + is_dyspnea_mod + is_brady_pnea + is_tachy_pnea +
                is_hypo_tension + is_hyper_tension + is_brady_tot + is_brady_mod +
                is_tachy_tot + is_tachy_mod) # n_abnormal_vitals
    X[:, 36] = (t_hr / 80.0) - (t_sbp / 120.0) # perfusion_gap
    X[:, 37] = np.clip(spo2_min - 90.0, -20.0, 20.0) # resp_reserve
    X[:, 38] = cc_bd * (100.0 - t_o2) # bd_x_o2_deficit
    X[:, 39] = cc_bd * is_tachy_pnea   # bd_x_tachypnea
    X[:, 40] = age * (100.0 - t_o2)   # age_o2_interaction
    X[:, 41] = ((t_hr - 80.0) / 80.0) ** 2 # hr_dev_sq
    
    return X

FEATURES = [
    'age', 'cc_breathingdifficulty', 'gender', 'triage_vital_hr', 'triage_vital_sbp', 'triage_vital_dbp', 'triage_vital_rr',
    'pulse_min', 'resp_min', 'spo2_min', 'pulse_max', 'spo2_max',
    'is_dyspnea_total', 'is_dyspnea_moderate', 'is_bradypnea', 'is_tachypnea', 'is_hypotension', 'is_hypertension',
    'is_bradycardia_total', 'is_bradycardia_moderate', 'is_tachycardia_total', 'is_tachycardia_moderate',
    'hr_range', 'rr_range', 'spo2_range', 'sbp_range',
    'shock_index', 'hr_mid_to_triage', 'sbp_mid_to_triage', 'rr_mid_to_triage', 'spo2_mid_to_triage',
    'rox_index', 'spo2_drop_ratio', 'hr_instability_ratio', 'bif',
    'n_abnormal_vitals', 'perfusion_gap', 'resp_reserve', 'bd_x_o2_deficit', 'bd_x_tachypnea',
    'age_o2_interaction', 'hr_dev_sq'
]

X = build_42_feature_matrix(raw_mat_in)

# 3-Class Mapping: RED (0: ESI 1-2), YELLOW (1: ESI 3), GREEN (2: ESI 4-5)
y = np.where(esi <= 2, 0, np.where(esi == 3, 1, 2))
LABELS = ['RED', 'YELLOW', 'GREEN']
TARGET_RED_SENSITIVITY = 0.90

# Model Hyperparameters
PARAMS = dict(objective='multiclass', num_class=3, learning_rate=0.05, num_leaves=31,
              feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
              min_child_samples=50, n_estimators=300, verbosity=-1,
              random_state=42, n_jobs=-1)

itr, itmp = train_test_split(np.arange(len(y)), test_size=0.30, stratify=y,
                             random_state=42)
iva, ite = train_test_split(itmp, test_size=0.50, stratify=y[itmp], random_state=42)

print(f"Extracted {len(FEATURES)} Features Matrix: {X.shape}")
print(f"Dataset Partition: rows={len(y)}  train={len(itr)}  val={len(iva)}  test={len(ite)}")
for name, n in zip(LABELS, np.bincount(y)):
    print(f'  {name:<7}{n:>8}  {100 * n / len(y):.1f}%')

In [ ]:
# ---------------------------------------------------------------------------
# Step 3: Train Multiclass LightGBM Model on 42 Curated Features
# ---------------------------------------------------------------------------
model = lgb.LGBMClassifier(**PARAMS)
model.fit(X[itr], y[itr], eval_set=[(X[iva], y[iva])],
          callbacks=[lgb.early_stopping(40, verbose=False)])

dump = model.booster_.dump_model()
n_nodes = sum(t['num_leaves'] for t in dump['tree_info'])
print(f'best_iteration={model.best_iteration_}  trees={len(dump["tree_info"])}  '
      f'nodes={n_nodes}  ~{n_nodes * 16 / 1024:.0f} KB in .rodata')

p_val = model.predict_proba(X[iva])
p_test = model.predict_proba(X[ite])
argmax = p_test.argmax(1)

recs_plain = recall_score(y[ite], argmax, average=None)
print(f'\nplain argmax benchmark (42 Features):')
print(f'  Balanced Accuracy     = {balanced_accuracy_score(y[ite], argmax):.4f}')
print(f'  Overall Accuracy      = {accuracy_score(y[ite], argmax):.4f}')
print(f'  RED Sensitivity/Recall    = {recs_plain[0]:.4f} ({LABELS[0]})')
print(f'  YELLOW Sensitivity/Recall = {recs_plain[1]:.4f} ({LABELS[1]})')
print(f'  GREEN Sensitivity/Recall  = {recs_plain[2]:.4f} ({LABELS[2]})')
print(f'  Macro Sensitivity/Recall  = {np.mean(recs_plain):.4f}')
print(f'  Macro F1-Score        = {f1_score(y[ite], argmax, average="macro"):.4f}')
print(f'  Macro ROC-AUC         = {roc_auc_score(y[ite], p_test, multi_class="ovr", average="macro"):.4f}')

In [ ]:
# ---------------------------------------------------------------------------
# Step 4: Threshold Optimization for Clinical Safety
# ---------------------------------------------------------------------------
def apply_threshold(p, t):
    return np.where(p[:, 0] >= t, 0, np.where(p[:, 1] >= p[:, 2], 1, 2))


def safety(y_true, y_pred):
    c = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    # Sensitivity == Recall per class
    rec_red    = c[0, 0] / c[0].sum() if c[0].sum() > 0 else 0.0
    rec_yellow = c[1, 1] / c[1].sum() if c[1].sum() > 0 else 0.0
    rec_green  = c[2, 2] / c[2].sum() if c[2].sum() > 0 else 0.0
    
    return dict(red_sensitivity=rec_red,
                red_recall=rec_red,
                yellow_sensitivity=rec_yellow,
                yellow_recall=rec_yellow,
                green_sensitivity=rec_green,
                green_recall=rec_green,
                macro_recall=np.mean([rec_red, rec_yellow, rec_green]),
                macro_sensitivity=np.mean([rec_red, rec_yellow, rec_green]),
                undertriage=c[0, 2] / c[0].sum() if c[0].sum() > 0 else 0.0,
                overtriage=(c[1, 0] + c[2, 0]) / (c[1].sum() + c[2].sum()) if (c[1].sum() + c[2].sum()) > 0 else 0.0,
                overtriage_green_only=c[2, 0] / c[2].sum() if c[2].sum() > 0 else 0.0,
                balanced_accuracy=balanced_accuracy_score(y_true, y_pred),
                accuracy=accuracy_score(y_true, y_pred))


grid = np.round(np.arange(0.50, 0.02, -0.01), 2)
hit = [t for t in grid
       if safety(y[iva], apply_threshold(p_val, t))['red_sensitivity']
       >= TARGET_RED_SENSITIVITY]
THRESHOLD = float(hit[0]) if hit else 0.50
print(f'RED threshold chosen on validation: {THRESHOLD}')

print('\n  t      RED_Sens/Rec  YELLOW_Sens/Rec  GREEN_Sens/Rec  undertriage  overtriage  Balanced_Acc  Accuracy')
for t in sorted({0.50, 0.40, 0.30, 0.25, 0.20, THRESHOLD, 0.15, 0.10}, reverse=True):
    s = safety(y[ite], apply_threshold(p_test, t))
    tag = '  <-- selected' if t == THRESHOLD else ''
    print(f'  {t:<6.2f} {s["red_sensitivity"]:^13.4f} {s["yellow_sensitivity"]:^16.4f} '
          f'{s["green_sensitivity"]:^15.4f} {s["undertriage"]:^12.4f} {s["overtriage"]:^11.4f} '
          f'{s["balanced_accuracy"]:^13.4f} {s["accuracy"]:.4f}{tag}')

In [ ]:
# ---------------------------------------------------------------------------
# Step 5: Holdout Report & Confusion Matrix Visualization
# ---------------------------------------------------------------------------
pred = apply_threshold(p_test, THRESHOLD)
s = safety(y[ite], pred)
auc = roc_auc_score(y[ite], p_test, multi_class='ovr', average='macro')
cm = confusion_matrix(y[ite], pred, labels=[0, 1, 2])

print("========================================================================")
print('           HOLDOUT TEST -- 3-CLASS DISASTER TRIAGE (42 FEATURES)')
print("========================================================================")
print(f'  Balanced Accuracy (at threshold) : {s["balanced_accuracy"]:.4f}')
print(f'  Balanced Accuracy (plain argmax) : {balanced_accuracy_score(y[ite], argmax):.4f}')
print(f'  Overall Accuracy (at threshold)  : {s["accuracy"]:.4f}')
print(f'  Overall Accuracy (plain argmax)  : {accuracy_score(y[ite], argmax):.4f}')
print(f'  Macro ROC-AUC                    : {auc:.4f}')
print("------------------------------------------------------------------------")
print("  Per-Class Sensitivity (Recall) Breakdown:")
print(f'    * RED Sensitivity / Recall     : {s["red_sensitivity"]:.4f} (Class 0: ESI 1-2)')
print(f'    * YELLOW Sensitivity / Recall  : {s["yellow_sensitivity"]:.4f} (Class 1: ESI 3)')
print(f'    * GREEN Sensitivity / Recall   : {s["green_sensitivity"]:.4f} (Class 2: ESI 4-5)')
print(f'    * Macro Sensitivity / Recall   : {s["macro_recall"]:.4f}')
print("------------------------------------------------------------------------")
print("  Clinical Safety Doctrines:")
print(f'    * Undertriage (RED -> GREEN)   : {s["undertriage"]:.4f}    (Doctrine standard: < 0.05)')
print(f'    * Overtriage (non-RED -> RED)  : {s["overtriage"]:.4f}    (Doctrine standard: <= 0.50)')
print(f'      - of which GREEN only        : {s["overtriage_green_only"]:.4f}')
print("========================================================================\n")

import matplotlib.pyplot as plt

norm = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.imshow(norm, cmap='Greens', vmin=0, vmax=1)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{cm[i, j]}\n({100 * norm[i, j]:.1f}%)', ha='center',
                va='center', color='white' if norm[i, j] > 0.55 else 'black')
ax.set_xticks(range(3), LABELS)
ax.set_yticks(range(3), LABELS)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Disaster Triage Holdout (42 Features, RED threshold {THRESHOLD})\nBalanced Acc: {s["balanced_accuracy"]:.4f} | Macro Recall: {s["macro_recall"]:.4f}')
plt.tight_layout()
os.makedirs(f'{ROOT}/plots', exist_ok=True)
plt.savefig(f'{ROOT}/plots/disaster_triage_confusion_matrix.png', dpi=200,
            bbox_inches='tight')
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Step 6: Secondary Evaluation (5-Class ESI Benchmark)
# ---------------------------------------------------------------------------
P5 = dict(PARAMS, num_class=5)
m5 = lgb.LGBMClassifier(**P5)
m5.fit(X[itr], esi[itr] - 1, eval_set=[(X[iva], esi[iva] - 1)],
       callbacks=[lgb.early_stopping(40, verbose=False)])
p5 = m5.predict_proba(X[ite])
pr5 = p5.argmax(1) + 1

rec5 = recall_score(esi[ite], pr5, average=None)
print('5-class ESI (secondary, for literature comparison)')
print(f'  Balanced Accuracy : {balanced_accuracy_score(esi[ite], pr5):.4f}')
print(f'  Overall Accuracy  : {accuracy_score(esi[ite], pr5):.4f}')
print(f'  Macro Sensitivity : {np.mean(rec5):.4f}')
for c_i, r_val in enumerate(rec5, 1):
    print(f'    * ESI {c_i} Sensitivity / Recall : {r_val:.4f}')
print(f'  QWK               : {cohen_kappa_score(esi[ite], pr5, weights="quadratic"):.4f}')
print(f'  within +-1        : {np.mean(np.abs(esi[ite] - pr5) <= 1):.4f}')
print(f'  Macro ROC-AUC     : {roc_auc_score(esi[ite], p5, multi_class="ovr", average="macro"):.4f}')

In [ ]:
# ---------------------------------------------------------------------------
# Step 7: Export Production Bundle & Manifest
# ---------------------------------------------------------------------------
deploy = f'{ROOT}/deploy'
os.makedirs(deploy, exist_ok=True)

with open(f'{deploy}/disaster_triage_3class.pkl', 'wb') as f:
    pickle.dump(dict(model=model, threshold=THRESHOLD, features=FEATURES,
                     labels=LABELS), f)

manifest = dict(labels=LABELS, red_threshold=THRESHOLD, feature_order=FEATURES,
                n_features=len(FEATURES), n_nodes=n_nodes,
                best_iteration=int(model.best_iteration_),
                holdout={k: round(float(v), 4) for k, v in s.items()},
                holdout_macro_auc=round(float(auc), 4))
with open(f'{deploy}/disaster_triage_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

print(f'wrote {deploy}/disaster_triage_3class.pkl')
print(f'wrote {deploy}/disaster_triage_manifest.json')

In [ ]:
# ---------------------------------------------------------------------------
# Step 8: Runnable Safety & Production Verifications
# ---------------------------------------------------------------------------
assert s['red_sensitivity'] >= 0.90, f'RED sensitivity {s["red_sensitivity"]:.4f}'
assert s['undertriage'] < 0.05, f'undertriage {s["undertriage"]:.4f} breaks doctrine'
assert s['overtriage'] <= 0.50, f'overtriage {s["overtriage"]:.4f} breaks doctrine'
assert n_nodes * 16 <= 2 * 1024 * 1024, f'{n_nodes} nodes exceeds the flash budget'
assert len(FEATURES) == 42, f'expected 42 features, got {len(FEATURES)}'
assert len(FEATURES) == len(set(FEATURES)), 'duplicate feature name'
assert json.load(open(f'{deploy}/disaster_triage_manifest.json'))['feature_order'] \
    == FEATURES, 'manifest feature order does not match the trained model'
print('all checks passed')